# The International Patent Classification table (TLS209_APPLN_IPC)

Welcome to the **International Patent Plassification table** in PATSTAT, namely table TLS209_APPLN_IPC. The table contains all international patent classifications linked to the applications. The set of classifications linked to a single application is a de-duplicated merge of all classifications of the various publication instances linked to the specific application. Additionally, only the latest version of the IPC classifications is used. This means that the user does not have to worry about reclassifications because older applications will always be classified according to the latest IPC version.


Information on classification according to the International Patent Classification (IPC) can be found on the [WIPO website](https://www.wipo.int/classifications/ipc/en/).

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS209_APPLN_IPC

## APPLN_ID

This is the unique identifier for each application that allows to link this table to table TLS201.

In [2]:
# Import table TLS201
from epo.tipdata.patstat.database.models import TLS201_APPLN

show_join = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS209_APPLN_IPC.ipc_class_symbol
).join(
    TLS201_APPLN, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id
).limit(1000)

show_join_df = patstat.df(show_join)
show_join_df

,appln_id,appln_auth,ipc_class_symbol
0,17444413,EP,G11B 20/00
1,22393814,GB,C09B 23/14
2,582548046,CN,C25B 11/091
3,319377972,DE,G06K 7/10
4,528546250,CN,F26B 25/00
...,...,...,...
995,7486708,CN,A23G 4/00
996,25548634,JP,A23L 27/10
997,489933328,US,B01D 69/02
998,555842375,CN,B65G 43/08


## IPC_CLASS_SYMBOL

Classification symbol according to the International Patent Classification, eighth edition (entered into force 1 January 2006). It consists of up to 15 characters, including Latin letters, Arabic numbers, space and '/'. Some spaces may be required since the slash '/' is always on the 9th position.

Each application can be associated with more than one class symbol. Let's verify this by counting the `ipc_class_symbol` and grouping by `appln_id`. We show only those applications that have a count greater than one.

In [3]:
from sqlalchemy import func

symb_appln = db.query(
    TLS209_APPLN_IPC.appln_id,
    func.count(TLS209_APPLN_IPC.ipc_class_symbol).label('Number of symbols')
).group_by(
    TLS209_APPLN_IPC.appln_id
).having(
    func.count(TLS209_APPLN_IPC.ipc_class_symbol) > 1  # Consider only applications with more than 1 class symbol
).order_by(
    func.count(TLS209_APPLN_IPC.ipc_class_symbol).desc()
).limit(1000)

symb_appln_df = patstat.df(symb_appln)
symb_appln_df

,appln_id,Number of symbols
0,8418577,244
1,8154006,243
2,8161391,243
3,7921091,242
4,55664994,241
...,...,...
995,390235,128
996,315635470,128
997,8418688,128
998,42234563,128


In [4]:
check = db.query(
    TLS209_APPLN_IPC.appln_id,
    TLS209_APPLN_IPC.ipc_class_symbol
).filter(
    TLS209_APPLN_IPC.appln_id == 8161391
)

check_df = patstat.df(check)
check_df

,appln_id,ipc_class_symbol
0,8161391,H04W 72/12
1,8161391,H04B 7/185
2,8161391,H04N 5/232
3,8161391,H04L 25/02
4,8161391,H03M 13/00
...,...,...
238,8161391,H04L 27/18
239,8161391,H04W 12/10
240,8161391,G09C 1/00
241,8161391,G06F 13/00


In [5]:
from sqlalchemy import select

sub = check.subquery()

duplicates = db.query(
    func.count(sub.c.appln_id).label('Number of duplicates'),
    sub.c.ipc_class_symbol
).filter(
    sub.c.appln_id == 8161391
).group_by(
    sub.c.ipc_class_symbol
).order_by(
    func.count(sub.c.appln_id)
)

duplicates_df = patstat.df(duplicates)
duplicates_df

,Number of duplicates,ipc_class_symbol
0,1,G02B 26/10
1,1,H04W 4/12
2,1,G01S 1/02
3,1,C07C 67/52
4,1,G06F 13/362
...,...,...
238,1,H04M 1/724
239,1,G06F 3/14
240,1,H04M 1/715
241,1,G06F 11/10


The contrary is also true, i.e. the same symbol can be assigned to more than one application ID.

In [6]:
appln_symb = db.query(
    TLS209_APPLN_IPC.ipc_class_symbol,
    func.count(TLS209_APPLN_IPC.appln_id).label('Number of symbols')
).group_by(
    TLS209_APPLN_IPC.ipc_class_symbol
).having(
    func.count(TLS209_APPLN_IPC.appln_id) > 1  # Consider only applications with more than 1 class symbol
).order_by(
    func.count(TLS209_APPLN_IPC.appln_id).desc()
)

appln_symb_df = patstat.df(appln_symb)
appln_symb_df

,ipc_class_symbol,Number of symbols
0,A61P 35/00,754413
1,G06F 17/30,502454
2,A61P 43/00,490671
3,H04L 29/06,481703
4,A61P 29/00,397928
...,...,...
80942,C01G 45/1264,2
80943,A61K 6/68,2
80944,H04L 12/815,2
80945,H02K 15/125,2


## IPC_CLASS_LEVEL

This attribute denotes whether an authority classified either in the full IPC, in main groups or in subclasses only. The domain of this attribute is 1 character.
* A = classification in the full IPC
* C = classification in main groups only
* S = classification in subclasses only

Let's find out which is the most frequent type of classifications.

In [7]:
# Count the number of applications (appln_id) grouped by ipc_class_level
levels = db.query(
    TLS209_APPLN_IPC.ipc_class_level,
    func.count(TLS201_APPLN.appln_id).label('Total number')
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).group_by(
    TLS209_APPLN_IPC.ipc_class_level
).order_by(
    func.count(TLS201_APPLN.appln_id).desc()  # Rank according to the most frequent class level
)

levels_df = patstat.df(levels)
levels_df

,ipc_class_level,Total number
0,A,362357478
1,S,1674541
2,C,375984


## IPC_VERSION

This is the version of the IPC. First of all, we can check how many applications there are in PATSTAT.

In [8]:
num_versions = db.query(
    func.count(TLS209_APPLN_IPC.ipc_version.distinct()).label('Distinct versions')
)

num_versions = patstat.df(num_versions)
num_versions = num_versions['Distinct versions'].item()
print("There are "+str(num_versions)+" distinct versions in PATSTAT.")

There are 22 distinct versions in PATSTAT.


Now let's find out which is the most frequent version.

In [9]:
version = db.query(
    TLS209_APPLN_IPC.ipc_version,
    func.count(TLS201_APPLN.appln_id).label('Number of applications')
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).group_by(
    TLS209_APPLN_IPC.ipc_version
).order_by(
    func.count(TLS201_APPLN.appln_id).desc()
)

version_df = patstat.df(version)
version_df

,ipc_version,Number of applications
0,2006-01-01,312578163
1,2016-01-01,4567834
2,2018-01-01,4322544
3,2012-01-01,4279664
4,2022-01-01,4159323
5,2023-01-01,4012193
6,2014-01-01,3623016
7,2009-01-01,3428092
8,2013-01-01,3364328
9,2019-01-01,3242955


## IPC_VALUE

This attribute tells the value of the classification, i.e. the class symbol relating to the invention or to aspects not related to the invention but present in the application. The domain is 1 character.
* I = invention
* N = additional (non-Invention)

In [10]:
value = db.query(
    TLS209_APPLN_IPC.ipc_value,
    func.count(TLS201_APPLN.appln_id).label('Occurrencies')
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).group_by(
    TLS209_APPLN_IPC.ipc_value
).order_by(
    func.count(TLS201_APPLN.appln_id).desc()
)

value_df = patstat.df(value)
value_df

,ipc_value,Occurrencies
0,I,353059856
1,N,11348141
2,,6


## IPC_POSITION

Indicates the position of the class symbol in the sequence of classes that form the classification. For patent authorities (e.g. USPTO) where the law entails the concept of "first" class, the first class symbol in a list of class symbols is the main class. For other authorities, like the EPO, there is no meaning in the position; classes may be quoted in alphabetical order, for instance.

The domain is represented by 1 character.
* F = fist
* L = later
* space = unidentified

In [11]:
position = db.query(
    TLS209_APPLN_IPC.ipc_position,
    func.count(TLS201_APPLN.appln_id).label('Occurrencies')
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).group_by(
    TLS209_APPLN_IPC.ipc_position
).order_by(
    func.count(TLS201_APPLN.appln_id).desc()
)

position_df = patstat.df(position)
position_df

,ipc_position,Occurrencies
0,L,190862784
1,F,92611601
2,,80933618


## IPC_GENER_AUTH

This attribute indicates which authority generated the IPC. It can differ from the application authority.

Let's find the applications that do not have 'EP' as `appln_auth` but having 'EP' as `ipc_gener_auth`. We limit the result to the first 1000 results for sake of computation time.

In [12]:
clashes = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS209_APPLN_IPC.ipc_gener_auth
).join(
    TLS201_APPLN, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id
).filter(
    TLS201_APPLN.appln_auth != 'EP',
    TLS209_APPLN_IPC.ipc_gener_auth == 'EP'
).limit(1000)

clashes_df = patstat.df(clashes)
clashes_df

,appln_id,appln_auth,ipc_gener_auth
0,6636544,CN,EP
1,18639657,FR,EP
2,4292385,CA,EP
3,45256829,TW,EP
4,21914259,GB,EP
...,...,...,...
995,1412623,AU,EP
996,9018768,CZ,EP
997,45035400,TW,EP
998,494303855,ES,EP


Suppose that we want to know how many times the attributes `appln_auth` and `ipc_gener_auth` differ in the entire database.

In [13]:
count_clashes = db.query(
    func.count(TLS201_APPLN.appln_id).label('clashes_counting')
).select_from(
    TLS201_APPLN  # Use select_from to specify how to join the two tables an avoid an InvalidRequestError
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).filter(
    TLS201_APPLN.appln_auth != TLS209_APPLN_IPC.ipc_gener_auth
)

count_clashes_df = patstat.df(count_clashes)
count_clashes_df = count_clashes_df['clashes_counting'].item()
print("There are "+str(count_clashes_df)+" applications for which application authority and IPC generating authority differ.")

There are 123074642 applications for which application authority and IPC generating authority differ.


### Top generative authorities

Of these applications, which is the most frequent IPC generative authority?

In [14]:
most_gen_auth = db.query(
    TLS209_APPLN_IPC.ipc_gener_auth,
    func.count(TLS201_APPLN.appln_id).label('Number of occurrencies')
).join(
    TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
).filter(
    TLS201_APPLN.appln_auth != TLS209_APPLN_IPC.ipc_gener_auth
).group_by(
    TLS209_APPLN_IPC.ipc_gener_auth
).order_by(
    func.count(TLS201_APPLN.appln_id).desc()
)

most_gen_auth_df = patstat.df(most_gen_auth)
most_gen_auth_df

,ipc_gener_auth,Number of occurrencies
0,EP,79458420
1,JP,32189927
2,US,2912153
3,KR,2177798
4,RU,1682499
...,...,...
87,GH,2
88,CG,2
89,MK,2
90,SM,1


The top two generative authorities are the EPO and the Japan Patent Office. Notice that they classified one order of magnitude of applications more than the following authorities.